# NOTEBOOK : 05_data_split.ipynb
# ROLE     : Chronological train/val/test split, fit StandardScaler on train only, export splits & artifacts.
# INPUT    : ../../data/processed/features.csv
# OUTPUT   : ../../data/processed/{train.csv, val.csv, test.csv, live_2026.csv}
#            ../artifacts/models/match_scaler.joblib, ../artifacts/features_list.json

In [1]:
from pathlib import Path
import json
import joblib
import pandas as pd
from sklearn.preprocessing import StandardScaler

PROCESSED_DIR = Path("../../data/processed")
ARTIFACTS_DIR = Path("../artifacts")
MODELS_DIR = ARTIFACTS_DIR / "models"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(PROCESSED_DIR / "features.csv")
df["Match_Date"] = pd.to_datetime(df["Match_Date"])
df = df.sort_values(["Match_Date", "Home_Team"]).reset_index(drop=True)

print(f"Loaded features.csv: {df.shape[0]} rows, {df.shape[1]} columns")
print("Seasons present:", sorted(df["season"].unique()))

Loaded features.csv: 2322 rows, 46 columns
Seasons present: ['2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26', '2026-27']


In [2]:
# Identify exact chronological midpoint date of 2025-26 (after 190th match / GW 19)
s2526 = df[df["season"] == "2025-26"].sort_values("Match_Date")
mid_date_2526 = s2526.iloc[len(s2526) // 2 - 1]["Match_Date"]
print(f"2025-26 midpoint cutoff date: {mid_date_2526.date()}")

# Strict chronological splits
train_seasons = ["2020-21", "2021-22", "2022-23", "2023-24", "2024-25"]

train_df = df[df["season"].isin(train_seasons)].copy()
val_df = df[(df["season"] == "2025-26") & (df["Match_Date"] <= mid_date_2526)].copy()
test_df = df[
    ((df["season"] == "2025-26") & (df["Match_Date"] > mid_date_2526))
    | (df["season"] == "2026-27")
].copy()
live_df = df[df["season"] == "2026-27"].copy()

# Verify strict date ordering across splits
assert train_df["Match_Date"].max() < val_df["Match_Date"].min(), "Leakage: train overlaps val!"
assert val_df["Match_Date"].max() < test_df["Match_Date"].min(), "Leakage: val overlaps test!"

print(f"TRAIN : {len(train_df):4d} rows | {train_df['Match_Date'].min().date()} -> {train_df['Match_Date'].max().date()}")
print(f"VAL   : {len(val_df):4d} rows | {val_df['Match_Date'].min().date()} -> {val_df['Match_Date'].max().date()}")
print(f"TEST  : {len(test_df):4d} rows | {test_df['Match_Date'].min().date()} -> {test_df['Match_Date'].max().date()}")
print(f"LIVE  : {len(live_df):4d} rows | {live_df['Match_Date'].min().date()} -> {live_df['Match_Date'].max().date()}")

2025-26 midpoint cutoff date: 2026-01-01
TRAIN : 1872 rows | 2020-10-03 -> 2025-05-25
VAL   :  190 rows | 2025-08-15 -> 2026-01-01
TEST  :  260 rows | 2026-01-03 -> 2026-10-19
LIVE  :   70 rows | 2026-08-21 -> 2026-10-19


In [3]:
META_COLS = ["season", "Match_Date", "Home_Team", "Away_Team", "Full_Time_Result", "target"]
FEATURE_COLS = [c for c in df.columns if c not in META_COLS]

# Fit StandardScaler ONLY on train_df
scaler = StandardScaler()
scaler.fit(train_df[FEATURE_COLS])

# Save scaler and feature list
scaler_path = MODELS_DIR / "match_scaler.joblib"
features_list_path = ARTIFACTS_DIR / "features_list.json"

joblib.dump(scaler, scaler_path)
with open(features_list_path, "w", encoding="utf-8") as f:
    json.dump(FEATURE_COLS, f, indent=2)

print(f"Saved scaler to {scaler_path}")
print(f"Saved {len(FEATURE_COLS)} feature names to {features_list_path}")

Saved scaler to ..\artifacts\models\match_scaler.joblib
Saved 40 feature names to ..\artifacts\features_list.json


In [4]:
for name, split_df in [
    ("train.csv", train_df),
    ("val.csv", val_df),
    ("test.csv", test_df),
    ("live_2026.csv", live_df),
]:
    out_df = split_df.copy()
    out_df["Match_Date"] = out_df["Match_Date"].dt.strftime("%Y-%m-%d")
    out_path = PROCESSED_DIR / name
    out_df.to_csv(out_path, index=False)
    print(f"Saved {out_path} ({len(out_df)} rows)")

Saved ..\..\data\processed\train.csv (1872 rows)
Saved ..\..\data\processed\val.csv (190 rows)
Saved ..\..\data\processed\test.csv (260 rows)
Saved ..\..\data\processed\live_2026.csv (70 rows)
